<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/Part2_NetworkVariations_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2, Notebook 5, Network Depth and Skip Connections

In this notebook I investigate two coupled architectural choices, network depth (how many conv blocks before the FC head) and skip connections (ResNet-style residuals). They're paired in one notebook because deep networks without skip connections suffer vanishing gradients, so the cleanest experiment is to compare deep variants with and without skips against the baseline.

This time I run on both tasks. Depth interacts with overfitting (more parameters, easier to overfit), and the two tasks behave very differently on that front (regression trained clean across 20 epochs while classification overfit at epoch 7). So the same architectural change can lead to genuinely different conclusions, which is worth seeing.

Reduced the amount of epochs to experiment faster

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/AppliedDL/face_age.zip" /content/
!cp -r "/content/drive/MyDrive/Colab Notebooks/AppliedDL/data_splits" /content/
!unzip -q /content/face_age.zip -d /content/

## 2. Imports

Same as notebook 4.



In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

device: cuda
gpu: NVIDIA L4


## 3. Dataset class

Same `FaceAgeDataset` carried over.

In [4]:
CATEGORIES = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


class FaceAgeDataset(Dataset):
    def __init__(self, csv_path, task, transform=None):
        self.df = pd.read_csv(csv_path)
        self.task = task
        self.transform = transform
        self.paths = ["/content/" + p for p in self.df["path"]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)

        if self.task == "regression":
            label = torch.tensor(self.df.iloc[idx]["age"], dtype=torch.float32)
        else:
            cat = self.df.iloc[idx]["age_category"]
            label = torch.tensor(CAT_TO_IDX[cat], dtype=torch.long)

        return img, label

## 4. Transforms and dataloaders

Same minimal pipeline, both tasks active this notebook.

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds_reg = FaceAgeDataset("/content/data_splits/train.csv", "regression", transform)
val_ds_reg   = FaceAgeDataset("/content/data_splits/val.csv",   "regression", transform)

train_ds_cls = FaceAgeDataset("/content/data_splits/train.csv", "classification", transform)
val_ds_cls   = FaceAgeDataset("/content/data_splits/val.csv",   "classification", transform)

BATCH_SIZE = 64

train_loader_reg = DataLoader(train_ds_reg, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, persistent_workers=True)
val_loader_reg   = DataLoader(val_ds_reg,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True, persistent_workers=True)

train_loader_cls = DataLoader(train_ds_cls, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, persistent_workers=True)
val_loader_cls   = DataLoader(val_ds_cls,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True, persistent_workers=True)

print(f"train: {len(train_ds_reg)}    val: {len(val_ds_reg)}")

train: 7320    val: 1464


## 5. Train and eval functions


In [6]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    n_samples = 0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        out = model(imgs)
        if out.shape[-1] == 1:
            out = out.squeeze(-1)
        loss = loss_fn(out, lbls)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)

    return total_loss / n_samples


def evaluate(model, loader, loss_fn, task):
    model.eval()
    total_loss = 0
    n_samples = 0
    correct = 0
    abs_error_sum = 0

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            if out.shape[-1] == 1:
                out = out.squeeze(-1)
            loss = loss_fn(out, lbls)

            total_loss += loss.item() * imgs.size(0)
            n_samples += imgs.size(0)

            if task == "regression":
                abs_error_sum += (out - lbls).abs().sum().item()
            else:
                preds = out.argmax(dim=1)
                correct += (preds == lbls).sum().item()

    avg_loss = total_loss / n_samples
    if task == "regression":
        metric = abs_error_sum / n_samples
    else:
        metric = correct / n_samples

    return avg_loss, metric

## 6. Variant runner

Same wrapper as notebook 4, defaults tightened to max 12 epochs and patience 3 for faster sweeps.

In [7]:
def run_variant(model, train_loader, val_loader, loss_fn, task,
                max_epochs=12, patience=3, lr=1e-3, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    if task == "regression":
        best_metric = float("inf")
        better = lambda new, best: new < best
    else:
        best_metric = -float("inf")
        better = lambda new, best: new > best

    history = {"train_loss": [], "val_loss": [], "val_metric": []}
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_metric = evaluate(model, val_loader, loss_fn, task)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_metric"].append(val_metric)

        if better(val_metric, best_metric):
            best_metric = val_metric
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
            marker = " (best)"
        else:
            epochs_no_improve += 1
            marker = ""

        if verbose:
            print(f"  epoch {epoch:2d}    train_loss {train_loss:7.4f}    "
                  f"val_loss {val_loss:7.4f}    val_metric {val_metric:.4f}{marker}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"  stopped early at epoch {epoch}, no improvement for {patience} epochs")
            break

    model.load_state_dict(best_state)
    return {"best_metric": best_metric, "best_state": best_state, "history": history}

## 7. Depth + skip CNN

One model, two arguments, `n_blocks` and `use_skip`. Each block holds two convs, optional residual with a 1x1 projection to match channels, then max pool. Channel progression doubles per block from 32. FC head sized dynamically from the final feature map.

In [10]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_skip):
        super().__init__()
        self.use_skip = use_skip
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        # 1x1 projection so input channels match output channels for the skip add
        if use_skip:
            self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        self.pool = nn.MaxPool2d(2)
        self.act = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.act(self.conv1(x))
        out = self.conv2(out)
        if self.use_skip:
            out = out + self.proj(identity)
        out = self.act(out)
        out = self.pool(out)
        return out


class DepthSkipCNN(nn.Module):
    def __init__(self, num_outputs, n_blocks=3, use_skip=False):
        super().__init__()
        # channel progression, doubles per block from 32
        channels = [32 * (2 ** i) for i in range(n_blocks)]
        in_chs = [3] + channels[:-1]

        self.blocks = nn.ModuleList([
            ResBlock(in_chs[i], channels[i], use_skip) for i in range(n_blocks)
        ])

        # spatial size halves per block, input is 200
        spatial = 200 // (2 ** n_blocks)
        flat = channels[-1] * spatial * spatial

        self.fc1 = nn.Linear(flat, 128)
        self.fc2 = nn.Linear(128, num_outputs)
        self.act = nn.ReLU()

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x


# sanity check the four variants we will actually run
for n_blocks, use_skip in [(3, False), (4, False), (4, True), (5, True)]:
    m = DepthSkipCNN(num_outputs=1, n_blocks=n_blocks, use_skip=use_skip).to(device)
    out = m(torch.randn(2, 3, 200, 200).to(device))
    params = sum(p.numel() for p in m.parameters())
    print(f"n_blocks {n_blocks}    skip {use_skip}    out {tuple(out.shape)}    params {params:,}")

n_blocks 3    skip False    out (2, 1)    params 10,527,265
n_blocks 4    skip False    out (2, 1)    params 5,891,105
n_blocks 4    skip True    out (2, 1)    params 5,934,689
n_blocks 5    skip True    out (2, 1)    params 7,246,945


## 8. Variants

Four variants, the canonical depth + skip story.

- `b3_noskip`, three blocks no skip, the reference point for this notebook
- `b4_noskip`, deeper, no skip, asks whether more depth alone helps
- `b4_skip`, same depth with skip, isolates the skip effect
- `b5_skip`, deepest with skip, asks whether skip lets us push depth further

In [11]:
variants = [
    {"name": "b3_noskip", "n_blocks": 3, "use_skip": False},
    {"name": "b4_noskip", "n_blocks": 4, "use_skip": False},
    {"name": "b4_skip",   "n_blocks": 4, "use_skip": True},
    {"name": "b5_skip",   "n_blocks": 5, "use_skip": True},
]

for v in variants:
    print(f"{v['name']:12s}  n_blocks {v['n_blocks']}  skip {v['use_skip']}")

b3_noskip     n_blocks 3  skip False
b4_noskip     n_blocks 4  skip False
b4_skip       n_blocks 4  skip True
b5_skip       n_blocks 5  skip True


## 9. Regression run

Four variants on regression. MSE loss, MAE tracked.

In [12]:
import time

results_reg = {}
loss_fn = nn.MSELoss()

for v in variants:
    print(f"\n=== {v['name']} (n_blocks {v['n_blocks']}, skip {v['use_skip']}) ===")
    t0 = time.time()

    model = DepthSkipCNN(num_outputs=1, n_blocks=v["n_blocks"], use_skip=v["use_skip"]).to(device)
    result = run_variant(model, train_loader_reg, val_loader_reg, loss_fn, "regression")

    elapsed = time.time() - t0
    results_reg[v["name"]] = result
    print(f"  best val MAE {result['best_metric']:.3f}    time {elapsed:.0f}s")


=== b3_noskip (n_blocks 3, skip False) ===
  epoch  1    train_loss 617.0834    val_loss 480.8953    val_metric 16.9169 (best)
  epoch  2    train_loss 330.7707    val_loss 250.4322    val_metric 11.5877 (best)
  epoch  3    train_loss 238.6514    val_loss 253.1931    val_metric 11.2380 (best)
  epoch  4    train_loss 183.0175    val_loss 195.0033    val_metric 9.8703 (best)
  epoch  5    train_loss 142.3912    val_loss 144.7942    val_metric 8.6103 (best)
  epoch  6    train_loss 119.3246    val_loss 169.6561    val_metric 9.2456
  epoch  7    train_loss 112.3827    val_loss 121.8242    val_metric 8.2044 (best)
  epoch  8    train_loss 84.7769    val_loss 109.1971    val_metric 7.5385 (best)
  epoch  9    train_loss 74.7306    val_loss 110.2083    val_metric 7.4364 (best)
  epoch 10    train_loss 66.3440    val_loss 99.6321    val_metric 7.0554 (best)
  epoch 11    train_loss 51.5408    val_loss 100.7328    val_metric 7.0328 (best)
  epoch 12    train_loss 46.6914    val_loss 113.084

## 10. Classification sweep

Same four variants on classification. CrossEntropy loss, accuracy tracked.

In [13]:
results_cls = {}
loss_fn = nn.CrossEntropyLoss()

for v in variants:
    print(f"\n=== {v['name']} (n_blocks {v['n_blocks']}, skip {v['use_skip']}) ===")
    t0 = time.time()

    model = DepthSkipCNN(num_outputs=7, n_blocks=v["n_blocks"], use_skip=v["use_skip"]).to(device)
    result = run_variant(model, train_loader_cls, val_loader_cls, loss_fn, "classification")

    elapsed = time.time() - t0
    results_cls[v["name"]] = result
    print(f"  best val acc {result['best_metric']:.4f}    time {elapsed:.0f}s")


=== b3_noskip (n_blocks 3, skip False) ===
  epoch  1    train_loss  1.5644    val_loss  1.2513    val_metric 0.4939 (best)
  epoch  2    train_loss  1.0862    val_loss  1.0388    val_metric 0.5779 (best)
  epoch  3    train_loss  0.8713    val_loss  0.9865    val_metric 0.5867 (best)
  epoch  4    train_loss  0.7080    val_loss  1.0043    val_metric 0.6107 (best)
  epoch  5    train_loss  0.5621    val_loss  1.0430    val_metric 0.6257 (best)
  epoch  6    train_loss  0.3929    val_loss  1.2708    val_metric 0.5772
  epoch  7    train_loss  0.2556    val_loss  1.5054    val_metric 0.6086
  epoch  8    train_loss  0.1614    val_loss  1.9131    val_metric 0.6086
  stopped early at epoch 8, no improvement for 3 epochs
  best val acc 0.6257    time 131s

=== b4_noskip (n_blocks 4, skip False) ===
  epoch  1    train_loss  1.8328    val_loss  1.5167    val_metric 0.4016 (best)
  epoch  2    train_loss  1.2765    val_loss  1.1803    val_metric 0.5082 (best)
  epoch  3    train_loss  1.0846